# Netflix Data Cleaning and Datetime Parsing

## Part 1 – Quantifying Missing Data

In [ ]:
import pandas as pd

netflix_titles = pd.read_csv("netflix_titles.csv")
netflix_titles.shape

In [ ]:
netflix_titles.isnull().sum()

In [ ]:
missing_pct = (netflix_titles.isnull().sum() / len(netflix_titles) * 100).round(2)
missing_pct.sort_values(ascending=False)

#### Part 1 Summary:

- The dataset has 8,807 rows and 12 columns.
- `director` has the worst data gap: 2,634 values are missing (29.91%), almost 1 in 3 titles.
- `country` (9.44%) and `cast` (9.37%) come next.
- `date_added`, `rating`, and `duration` are missing less than 0.2%, so their impact is very small.
- The other six columns have no missing values.

## Part 2 – Cleaning Strategies (Dropping vs. Imputing)

In [ ]:
# Strategy B experiment copy (made BEFORE Strategy A changes anything)
netflix_dropped = netflix_titles.copy()

In [ ]:
# Strategy A: fill missing directors with "Unknown"
netflix_titles["director"] = netflix_titles["director"].fillna("Unknown")

# Verify: missing count should now be 0
netflix_titles["director"].isnull().sum()

In [ ]:
# Strategy B: drop rows missing director OR cast (on the copy)
netflix_dropped = netflix_dropped.dropna(subset=["director", "cast"])

netflix_dropped.shape

In [ ]:
rows_before = len(netflix_titles)
rows_after = len(netflix_dropped)
rows_lost = rows_before - rows_after

print(f"Rows before: {rows_before}")
print(f"Rows after:  {rows_after}")
print(f"Rows lost:   {rows_lost} ({rows_lost / rows_before * 100:.2f}%)")

In [ ]:
print("Before:\n", netflix_titles["type"].value_counts())
print("\nAfter:\n", netflix_dropped["type"].value_counts())

#### Part 2 Summary:

**Strategy A (Imputation)** kept all 8,807 rows. The `director` missing count dropped from 2,634 to 0.

**Strategy B (Row Dropping)** left 5,700 rows.
**Rows lost:** 8,807 − 5,700 = **3,107 rows (35.28%)**.

The loss was not spread evenly across content types:

| Type | Before | After | Lost |
|---|---|---|---|
| Movie | 6,131 | 5,522 | 9.93% |
| TV Show | 2,676 | 178 | 93.35% |

**Scenario:** Imagine an analyst wants to compare Movies and TV Shows,
for example, "Does Netflix add more TV Shows than Movies over time?"
After Strategy B, TV Shows fall from about 30% of the catalog to only 3%.
The analysis would wrongly conclude that Netflix barely offers TV Shows.
This happens because TV Shows often have no single director, so dropping
rows with a missing `director` removes almost all of them.
In this case, filling with "Unknown" (Strategy A) is the safer choice.

## Part 3 – Processing Datetime Data

In [ ]:
netflix_titles.dtypes

In [ ]:
# Remove extra spaces, then convert text to datetime
netflix_titles["date_added"] = pd.to_datetime(netflix_titles["date_added"].str.strip())

netflix_titles.info()

In [ ]:
netflix_titles["day_added"] = netflix_titles["date_added"].dt.day

netflix_titles.head()

#### Part 3 Summary:

`.dtypes` showed that `date_added` was stored as text (e.g. "September 25, 2021").
The instructions expect `object`, but my output shows `str` because I'm using Pandas 3.0,
which gives text columns a dedicated string dtype. Either way, it is plain text, not a date.

I removed extra spaces with `.str.strip()` and converted the column with `pd.to_datetime()`.
`.info()` confirms it is now `datetime64[us]` (Pandas 3.0 uses microseconds by default).
The 10 missing dates became `NaT`.

Using the `.dt` accessor, I extracted the day of the month into a new column, `day_added`.
It shows as a float (e.g. 25.0) because the column contains `NaT` values.

# Technical Interview – Week 6
## Q1 – Extracting the Year Added

In [ ]:
# Demo: text sorting vs. datetime sorting
sample = pd.Series(["April 1, 2020", "January 5, 2021", "December 10, 2019"])

print("As text:\n", sample.sort_values().tolist())
print("\nAs datetime:\n", pd.to_datetime(sample).sort_values().dt.date.tolist())

In [ ]:
# Extract the year each title was added to Netflix
netflix_titles["year_added"] = netflix_titles["date_added"].dt.year

# Use it in analysis: how many titles were added each year?
netflix_titles["year_added"].value_counts().sort_index()

## Q2 – Hypothesis Test: Movies vs. TV Shows

**Question:** Do Netflix Movies and TV Shows differ in their release year?

- **H₀ (Null):** The mean release year of Movies equals the mean release year of TV Shows.
- **Hₐ (Alternative):** The mean release years are different.
- **Significance level:** α = 0.05
- **Test:** Independent two-sample t-test (`scipy.stats.ttest_ind`), comparing two separate groups.

In [ ]:
from scipy import stats

# Split release years into two groups
movies = netflix_titles[netflix_titles["type"] == "Movie"]["release_year"]
tv_shows = netflix_titles[netflix_titles["type"] == "TV Show"]["release_year"]

print(f"Movies:   n = {len(movies)}, mean release year = {movies.mean():.1f}")
print(f"TV Shows: n = {len(tv_shows)}, mean release year = {tv_shows.mean():.1f}")

# Independent two-sample t-test
t_stat, p_value = stats.ttest_ind(movies, tv_shows, equal_var=False)
print(f"\nt-statistic = {t_stat:.2f}")
print(f"p-value     = {p_value:.2e}")

### Q2 – Results and Interpretation

| Group | n | Mean release year |
|---|---|---|
| Movies | 6,131 | 2013.1 |
| TV Shows | 2,676 | 2016.6 |

- **t-statistic = −20.98**: negative because Movies have the lower (older) mean.
- **p-value = 3.71 × 10⁻⁹⁵**, far below α = 0.05.

**Decision:** Reject H₀. Movies and TV Shows on Netflix differ in their release years.
On average, Netflix's Movies are about 3.5 years older than its TV Shows.

**What the p-value means:** If there were truly no difference between the two groups,
the chance of seeing a gap this large by random chance would be almost zero.
It is *not* the probability that H₀ is true.

**Caveat:** With over 8,800 titles, even small differences become statistically significant.
Here the gap (about 3.5 years) is also large enough to matter in practice.
One possible reason: Netflix licenses many older movies, but most of its TV Shows are recent originals.